# 3. From preprocessed recordings to a trained model

This notebook takes the Siena dataset preprocessed in notebook 02 and

1. divides every recording into 4 s windows labelled seizure or background
   (metadata only: no EEG is read at this stage);
2. builds leave-one-patient-out (LOPO) folds, one per patient (14 folds);
3. creates PyTorch data loaders for every fold;
4. trains and tests EEGNet on every fold.

Training is the slow step: about an hour for the 14 folds on a CPU with 4–8 threads, and
much less on a GPU. Lower `EPOCHS` in section 5 for a quick check.

In [ ]:
from pathlib import Path
from time import perf_counter

import pandas as pd
import torch
from IPython.display import display
from torch import nn

from bids2seizureml.folds import CVPlan, create_fold_dataframes
from bids2seizureml.loaders import build_dataloaders_from_folds
from bids2seizureml.preprocessing import derivatives_dir_for_params
from bids2seizureml.segment import build_segments_dataframe, build_windowed_segments_dataframe

## 1. Find the preprocessed dataset

The folder name is computed from the preprocessing settings, so these must match the
settings used in notebook 02.

In [ ]:
DATASET_PATH = Path("../data/siena")
PREPROCESSING = {
    "montage": "Longitudinal",
    "notch_filter": 50.0,
    "target_sampling_freq": 256.0,
    "output_format": "h5",
}

DATA_DIR = derivatives_dir_for_params(dataset_path=DATASET_PATH, **PREPROCESSING)
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"{DATA_DIR.name} not found: run notebook 02 with the same settings first.")
print(f"Preprocessed dataset: derivatives/{DATA_DIR.name}")

## 2. Segments and windows

`build_segments_dataframe` reads the `events.tsv` file of every recording and divides the
recording into seizure and background intervals (segments). Events whose label starts with
`sz` are seizures; overlapping or touching seizures are merged.
`build_windowed_segments_dataframe` then cuts every segment into fixed-length windows that lie
entirely inside it, so each window is purely seizure or purely background. Each window also
stores its distance to the nearest seizure and to the nearest recording edge.

In [ ]:
WINDOWS = {
    "segment_length": 4.0,      # Window length in seconds.
    "overlap_seizure": 0.0,     # Overlap between consecutive seizure windows, in seconds.
    "overlap_background": 0.0,  # Overlap between consecutive background windows, in seconds.
}

start = perf_counter()
segments_df = build_segments_dataframe(str(DATA_DIR), centre="siena")
windows_df = build_windowed_segments_dataframe(segments_df, **WINDOWS)
print(f"Built in {perf_counter() - start:.1f} s")

print(f"{segments_df['subject'].nunique()} patients, {segments_df['events_path'].nunique()} recordings")
segment_length = segments_df["end_s"] - segments_df["start_s"]
display(pd.DataFrame({
    "segments": segments_df["label"].value_counts(),
    "mean segment length (s)": segment_length.groupby(segments_df["label"]).mean().round(1),
    "windows": windows_df["label"].value_counts(),
}))
display(windows_df[[
    "subject", "session", "run", "start_s", "end_s", "label",
    "nearest_seizure_distance_s", "nearest_recording_edge_distance_s",
]].head())

## 3. Leave-one-patient-out folds

Each fold holds out one patient for testing. Whole patients are assigned to the training or
test set before any window is selected, so no recording contributes to both.

All seizure windows are kept. Background windows are sampled at random for each patient, at
one background window per seizure window, after excluding windows less than 30 s from a
seizure or from a recording edge. The random seed is fixed per patient, so a patient
contributes the same windows to every fold in which they are in the training set.

With `eval_strategy="balanced_windows"`, the test set is sampled in the same way. The other
option, `"complete_recordings"`, is described at the end of the notebook.

In [ ]:
CV_PLAN = CVPlan(
    strategy="lopo",
    eval_strategy="balanced_windows",
    background_to_seizure_ratio=1.0,
    min_background_seizure_distance_s=30.0,
    min_background_recording_edge_distance_s=30.0,
    seed=42,
)
fold_data = create_fold_dataframes(windows_df, CV_PLAN)

print(f"{len(fold_data.folds)} folds")
display(fold_data.fold_summary_df.pivot(
    index="fold_id", columns="split", values=["n_seizure", "n_background"]
))

## 4. Data loaders

This is the only step that reads EEG. With `materialize="memory"`, every distinct window is
read once and shared by all folds. For larger datasets, `"memmap"` keeps the windows on disk,
and `None` reads them from the recordings each time they are needed.

Each window is returned as a `channels × samples` tensor with label 0 (background) or 1
(seizure), normalised when it is requested (`"global_zscore"`: one mean and standard deviation
over all channels of the window). Augmentation is off here; the comment shows an example. It
is applied to the training set only, as extra copies of the selected windows.

In [ ]:
LOADER_SETTINGS = {
    "batch_size": 32,
    "normalisation": "global_zscore",
    "augmentation": None,  # e.g. {"labels": "seizure", "copies": 1, "strategies": [{"name": "gaussian_noise", "snr_db": 20.0}]}
    "materialize": "memory",
    "seed": 42,
}

start = perf_counter()
loaders = build_dataloaders_from_folds(fold_data, **LOADER_SETTINGS)
print(f"Data loaders for {len(loaders)} folds built in {perf_counter() - start:.1f} s")

first_fold = next(iter(loaders))
x, y = next(iter(loaders[first_fold]["train"]))
print(f"First training batch of {first_fold}: x {tuple(x.shape)} (windows, channels, samples), y {y.tolist()[:8]} ...")

## 5. Train and test EEGNet on every fold

EEGNet-8,2 (Lawhern et al., 2018), trained for a fixed number of epochs with Adam, and tested
once on the held-out patient. A new model is trained for each fold. On a GPU, results can
differ slightly between runs because some cuDNN operations are not deterministic.

In [ ]:
EPOCHS = 150
LEARNING_RATE = 1e-4
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class EEGNet(nn.Module):
    """EEGNet-8,2 for two classes. Input shape: (batch, channels, samples)."""

    def __init__(self, n_channels, n_samples, f1=8, d=2, kernel_length=64, dropout=0.25):
        super().__init__()
        f2 = f1 * d
        self.features = nn.Sequential(
            nn.Conv2d(1, f1, (1, kernel_length), padding="same", bias=False),     # Temporal filters.
            nn.BatchNorm2d(f1),
            nn.Conv2d(f1, f2, (n_channels, 1), groups=f1, bias=False),            # Spatial filters.
            nn.BatchNorm2d(f2),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout),
            nn.Conv2d(f2, f2, (1, 16), padding="same", groups=f2, bias=False),   # Separable convolution.
            nn.Conv2d(f2, f2, (1, 1), bias=False),
            nn.BatchNorm2d(f2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(f2 * (n_samples // 32), 2)

    def forward(self, x):
        return self.classifier(self.features(x.unsqueeze(1)).flatten(1))

    @torch.no_grad()
    def apply_max_norm(self, spatial=1.0, classifier=0.25):
        """Max-norm constraints on the spatial filters and the classifier weights."""
        for layer, maximum in [(self.features[2], spatial), (self.classifier, classifier)]:
            norm = layer.weight.norm(p=2, dim=tuple(range(1, layer.weight.ndim)), keepdim=True)
            layer.weight.mul_(norm.clamp(max=maximum) / norm.clamp_min(1e-8))


def train_model(train_loader, n_channels, n_samples):
    torch.manual_seed(SEED)
    model = EEGNet(n_channels, n_samples).to(DEVICE)
    optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(EPOCHS):
        for x, y in train_loader:
            optimiser.zero_grad()
            loss = loss_fn(model(x.to(DEVICE)), y.to(DEVICE))
            loss.backward()
            optimiser.step()
            model.apply_max_norm()
    return model


@torch.no_grad()
def test_model(model, test_loader):
    """Confusion counts, with seizure as the positive class."""
    model.eval()
    labels, predictions = [], []
    for x, y in test_loader:
        labels.append(y)
        predictions.append(model(x.to(DEVICE)).argmax(dim=1).cpu())
    y, p = torch.cat(labels), torch.cat(predictions)
    return {
        "tp": int(((p == 1) & (y == 1)).sum()),
        "fn": int(((p == 0) & (y == 1)).sum()),
        "tn": int(((p == 0) & (y == 0)).sum()),
        "fp": int(((p == 1) & (y == 0)).sum()),
    }

In [ ]:
print(f"Training on {DEVICE} for {EPOCHS} epochs per fold")
results = []
for fold_id, fold_loaders in loaders.items():
    start = perf_counter()
    train_loader, test_loader = fold_loaders["train"], fold_loaders["test"]
    n_channels, n_samples = train_loader.dataset[0][0].shape
    model = train_model(train_loader, n_channels, n_samples)
    counts = test_model(model, test_loader)
    minutes = (perf_counter() - start) / 60
    results.append({
        "fold": fold_id,
        "train windows": len(train_loader.dataset),
        "test windows": len(test_loader.dataset),
        **counts,
        "minutes": minutes,
    })
    print(f"{fold_id}: {counts} ({minutes:.1f} min)")

results_df = pd.DataFrame(results).set_index("fold")
results_df["sensitivity"] = results_df["tp"] / (results_df["tp"] + results_df["fn"])
results_df["specificity"] = results_df["tn"] / (results_df["tn"] + results_df["fp"])
results_df["accuracy"] = (results_df["tp"] + results_df["tn"]) / results_df["test windows"]
display(results_df.round(3))

print("Mean and standard deviation over folds:")
display(results_df[["sensitivity", "specificity", "accuracy"]].agg(["mean", "std"]).T.round(3))

## Evaluating on complete recordings (optional)

A balanced test set is easy to interpret but does not reflect how rare seizures are. For a
more realistic evaluation, cover every recording with a regular grid of windows and use
`eval_strategy="complete_recordings"`:

```python
from bids2seizureml.segment import build_evaluation_windows_dataframe

evaluation_windows_df = build_evaluation_windows_dataframe(segments_df, segment_length=4.0)
fold_data = create_fold_dataframes(
    windows_df,
    CVPlan(strategy="lopo", eval_strategy="complete_recordings",
           min_background_seizure_distance_s=30.0, min_background_recording_edge_distance_s=30.0),
    evaluation_windows_df=evaluation_windows_df,
)
```

Training is unchanged, but each test set then contains every window of the held-out
recordings. Windows that are partly seizure have label −100 (ignored by
`nn.CrossEntropyLoss`, and by `test_model` above), and the last window of each recording is
padded with zeros. These test sets are much larger: use `materialize="memmap"` or `None`, and
`return_metadata=True` to get each window's seizure fraction and padding.